# Distilling Vibration-Analysis Expertise from Big MoE Teachers into a Small Student

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winkelermatthias/MachineLearningSamples-DeepLearningforPredictiveMaintenance/blob/claude/distill-moe-vibration-analysis-xxwyye/Code/4_distill_moe_vibration_expert.ipynb)

All the logic lives in the **`vibdistill` package** shipped in `vibdistill.zip` — **upload that zip to `/content`** (Files panel, drag & drop) before running. This notebook only orchestrates:

| Stage | What happens |
|---|---|
| **1. Synthetic data** | Massive in-memory generation of physically-motivated vibration spectral patterns (velocity + envelope spectra) with measured ground-truth labels: machine info, true running speed, fault type, ISO 20816-3 zone, bearing condition (`vibdistill.synth`) |
| **2. Teacher labeling** | **GLM-5.2** via [NVIDIA NIM](https://build.nvidia.com/z-ai/glm-5.2) and **Kimi K3** via the [Moonshot API](https://platform.moonshot.ai/) produce chain-of-thought diagnoses, **rejection-sampled against ground truth** (`vibdistill.teachers`, `vibdistill.verify`) |
| **3. Student fine-tuning** | QLoRA SFT of Qwen3-8B (A100) / Qwen3-4B (L4) on the accepted traces (`vibdistill.train`) |
| **4. Evaluation** | Base vs. distilled student on a held-out ground-truth set (`vibdistill.train.score`) |

The model must infer everything from the peak tables: the **true running speed** (only the nameplate range is given), the **fault**, the **bearing** (withheld in 25 % of samples), and **two severity grades** (ISO zone + bearing condition).

> **Runtime**: Colab **A100 40 GB** recommended; L4 works. Everything expensive is cached and resumable: corpus → `synthetic_corpus.jsonl`, teacher answers → `teacher_*.jsonl` (interrupt + re-run only fires missing calls), model load skipped when already in memory.
> **Cost/time knob**: `n_teacher` (default 4 000 ≈ 3–5 h at free-tier rate limits, both teachers concurrent; 400 for a smoke run).
> **Model names** (checked Aug 2026): `kimi-latest` was retired Jan 2026 → the flagship is `kimi-k3`; GLM-5.2 is `z-ai/glm-5.2` on NIM. The smoke-test cell verifies both live before anything expensive runs.

## 0 · Load the code + setup

In [ ]:
# Load the vibdistill package from the zip you placed in /content.
import os
import sys
import zipfile

ZIP = "/content/vibdistill.zip"
if not os.path.isdir("/content/vibdistill"):
    assert os.path.exists(ZIP), "Upload vibdistill.zip to /content first (Files panel, drag & drop)"
    zipfile.ZipFile(ZIP).extractall("/content")
    print("extracted", ZIP)
if "/content" not in sys.path:
    sys.path.insert(0, "/content")

import vibdistill
print("vibdistill", vibdistill.__version__, "loaded from", os.path.dirname(vibdistill.__file__))

In [ ]:
%%capture --no-stderr
!pip install -qU "openai>=1.40" "transformers>=4.46" "datasets>=2.20" "peft>=0.13" "trl>=0.12" "bitsandbytes>=0.44" accelerate scipy tqdm pandas matplotlib

In [ ]:
from vibdistill.config import Config, get_secret, teacher_registry

cfg = Config()          # all knobs: Config(n_teacher=400, epochs=3, out_dir=..., ...)
print(cfg)

# OPTIONAL — persist model downloads + caches across Colab runtimes:
# from google.colab import drive; drive.mount("/content/drive")
# os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
# cfg = Config(out_dir="/content/drive/MyDrive/distill_pdm")

TEACHERS = teacher_registry(
    nvidia_api_key=get_secret("NVIDIA_API_KEY"),      # nvapi-...  free tier at build.nvidia.com
    moonshot_api_key=get_secret("MOONSHOT_API_KEY"),  # sk-...     platform.moonshot.ai
)

In [ ]:
# Verify model ids are live + one real call per provider, BEFORE spending anything.
from vibdistill.teachers import smoke_test

smoke_test(TEACHERS)

## 1 · Synthetic spectral corpus (in memory, cached to disk)

In [ ]:
import numpy as np

from vibdistill.synth import build_corpus

rng = np.random.default_rng(cfg.seed)
dataset = build_corpus(cfg, rng)

eval_set = dataset[:cfg.n_eval]                              # ground-truth holdout
teacher_pool = dataset[cfg.n_eval:cfg.n_eval + cfg.n_teacher]
print(f"{len(dataset):,} samples | teacher pool {len(teacher_pool):,} | eval holdout {len(eval_set):,}")
print("\nExample report:\n")
print(dataset[3]["report"])
print("\nGround truth:", {k: dataset[3][k] for k in ("fault", "zone", "cond")},
      f"rpm={dataset[3]['rpm']:.0f}")

In [ ]:
from vibdistill.synth import demo_case
from vibdistill.viz import plot_distributions, plot_example

plot_distributions(dataset)
plot_example(demo_case("bearing_inner_race"))

## 2 · Teacher labeling — both providers concurrent, rejection-sampled

Interrupt-safe: answers checkpoint to `teacher_*.jsonl`; re-running only fires the missing calls. The cell after verifies every trace against ground truth (speed ±4 %, exact fault / zone / condition / bearing) — only fully-correct reasoning enters training.

In [ ]:
from vibdistill.teachers import run_teachers

await run_teachers(TEACHERS, teacher_pool, cfg.out_dir)

In [ ]:
from vibdistill.verify import collect

accepted, teacher_stats = collect(TEACHERS, dataset, cfg.out_dir, cfg.n_eval)

## 3 · Distill into the student (QLoRA) and evaluate

In [ ]:
from vibdistill.train import generate_batch, load_student, pick_student, score

# skipped on re-runs in the same session (restart the runtime for a fresh base model)
if "model" not in globals():
    STUDENT_MODEL, VRAM_GB = pick_student()
    model, tok = load_student(STUDENT_MODEL)

# baseline: the untuned student on the held-out set
eval_run = eval_set[:cfg.eval_run]
base_out = generate_batch(model, tok, [s["report"] for s in eval_run])
base_scores = score(eval_run, base_out)
print("base student:", {k: f"{v:.2f}" for k, v in base_scores.items()})

In [ ]:
from vibdistill.train import build_sft_dataset, train_student

ds = build_sft_dataset(accepted, cfg.seed)
print(ds)
trainer = train_student(model, tok, ds, cfg, VRAM_GB)

In [ ]:
trainer.model.gradient_checkpointing_disable()
trainer.model.config.use_cache = True
tuned_out = generate_batch(trainer.model, tok, [s["report"] for s in eval_run])
tuned_scores = score(eval_run, tuned_out)
print("distilled student:", {k: f"{v:.2f}" for k, v in tuned_scores.items()})

In [ ]:
from vibdistill.viz import plot_comparison

plot_comparison(base_scores, tuned_scores, base_label=f"Base {STUDENT_MODEL.split('/')[-1]}")
if teacher_stats is not None:
    print("\nteacher accuracy on their own pool (for reference):")
    print(teacher_stats)

In [ ]:
from vibdistill.train import save_adapter

final_dir = save_adapter(trainer, tok, cfg, STUDENT_MODEL)

# Optional: copy to Drive so it survives the runtime
# from google.colab import drive; drive.mount("/content/drive")
# !cp -r {final_dir} /content/drive/MyDrive/

# Optional: merge to a standalone bf16 model for vLLM / GGUF export
# import torch
# from peft import PeftModel
# from transformers import AutoModelForCausalLM
# base = AutoModelForCausalLM.from_pretrained(STUDENT_MODEL, torch_dtype=torch.bfloat16, device_map="cpu")
# merged = PeftModel.from_pretrained(base, final_dir).merge_and_unload()
# merged.save_pretrained("/content/vib-expert-merged"); tok.save_pretrained("/content/vib-expert-merged")

## Where to take this next

- **Close the sim-to-real gap**: render real spectra (CWRU, Paderborn, NASA IMS, this repo's C-MAPSS) into the same report format; without simulator ground truth, fall back to teacher-consensus voting between GLM-5.2 and Kimi K3.
- **Harden the curriculum**: multi-fault samples, resonance cases, 60 Hz fleets, variable-speed sweeps, "insufficient data" as a legitimate verdict.
- **Go beyond SFT**: the rejected traces are ready-made DPO negatives; or on-policy distillation (GKD in TRL) against a locally-served teacher.
- **Deploy**: merge the adapter and serve with vLLM, or export GGUF for edge boxes.

**Caveats**: simplified ISO 20816-3 group tables; synthetic accuracy measures *distillation transfer*, not production readiness. All logic is editable in `/content/vibdistill/*.py` — after editing, `import importlib, vibdistill.synth; importlib.reload(vibdistill.synth)` or restart the runtime.